# RL Lab 1

#### Required libraries

In [4]:
# If needed in a fresh environment:

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym


## Concept recap
Agent interacts with env; at each step: state → action → reward → next state; an episode ends at terminal. Return is cumulative reward; γ decides how much we care about future.

### 1. Warm-up: Explore FrozenLake (states/actions/episode)

In [5]:
env = gym.make("FrozenLake-v1", is_slippery=True)  # stochastic transitions
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

obs, info = env.reset()
print("Initial state:", obs, info)


Observation space: Discrete(16)
Action space: Discrete(4)
Initial state: 0 {'prob': 1}


Run 1 random episode and print the trajectory (s, a, r, s’).

In [6]:
def run_random_episode(env, max_steps=100):
    obs, info = env.reset()
    traj = []
    total_reward = 0.0
    for t in range(max_steps):
        a = env.action_space.sample()
        next_obs, r, terminated, truncated, info = env.step(a)
        traj.append((obs, a, r, next_obs))
        total_reward += r
        obs = next_obs
        if terminated or truncated:
            break
    return traj, total_reward

traj, G = run_random_episode(env)
print("Episode length:", len(traj), "Return:", G)
print("First 10 transitions:", traj[:10])


Episode length: 15 Return: 0.0
First 10 transitions: [(0, np.int64(0), 0, 0), (0, np.int64(0), 0, 0), (0, np.int64(3), 0, 0), (0, np.int64(2), 0, 1), (1, np.int64(2), 0, 2), (2, np.int64(2), 0, 2), (2, np.int64(1), 0, 1), (1, np.int64(0), 0, 0), (0, np.int64(2), 0, 0), (0, np.int64(1), 0, 0)]


#### Questions:

What is the state in FrozenLake? What is an action?

When does an episode end in this environment?

Why might the same action from the same state lead to different outcomes (stochasticity / transition probability)?

### 2. Return and discount factor (γ) with a simple reward list

In [7]:
def discounted_return(rewards, gamma):
    G = 0.0
    for t, r in enumerate(rewards):
        G += (gamma**t) * r
    return G

rewards = [1, 1, -1, 1]
for gamma in [0.2, 0.9, 1.0]:
    print(gamma, discounted_return(rewards, gamma))


0.2 1.168
0.9 1.819
1.0 2.0


#### Exercise:

Create two reward sequences with the same sum but different timing (early vs late rewards).

Compare discounted returns for γ=0.2 vs γ=0.9.

#### Questions:

In your own words, what changes when γ is small vs large?

### 3. Tiny GridWorld (A, B, C terminal)

This implements a minimal deterministic grid:

- States: A (top-left), B (top-right), C (bottom-right, terminal), X blocked (bottom-left)

- Actions: 0=LEFT, 1=DOWN, 2=RIGHT, 3=UP

- Reward: normal move = -1, moving into C = +10

- γ will be used later

In [8]:
from dataclasses import dataclass

LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3
ACTIONS = [LEFT, DOWN, RIGHT, UP]
ACTION_NAMES = {LEFT:"L", DOWN:"D", RIGHT:"R", UP:"U"}

@dataclass(frozen=True)
class TinyGrid:
    # 2x2 with one blocked cell:
    # (0,0)=A, (0,1)=B
    # (1,0)=X blocked, (1,1)=C terminal
    terminal = (1,1)
    blocked = (1,0)
    start_positions = {(0,0):"A", (0,1):"B", (1,1):"C"}

    def states(self):
        return [(0,0),(0,1),(1,1)]  # exclude blocked

    def is_terminal(self, s):
        return s == self.terminal

    def step(self, s, a):
        if self.is_terminal(s):
            return s, 0.0, True

        r, c = s
        nr, nc = r, c
        if a == LEFT:  nc -= 1
        if a == RIGHT: nc += 1
        if a == UP:    nr -= 1
        if a == DOWN:  nr += 1

        # boundaries
        if nr < 0 or nr > 1 or nc < 0 or nc > 1:
            nr, nc = r, c

        # blocked
        if (nr, nc) == self.blocked:
            nr, nc = r, c

        s2 = (nr, nc)

        # rewards
        if s2 == self.terminal:
            reward = 10.0
            done = True
        else:
            reward = -1.0
            done = False

        return s2, reward, done

grid = TinyGrid()
print("States:", grid.states())
print("From A, DOWN ->", grid.step((0,0), DOWN))
print("From B, DOWN ->", grid.step((0,1), DOWN))  # into C


States: [(0, 0), (0, 1), (1, 1)]
From A, DOWN -> ((0, 0), -1.0, False)
From B, DOWN -> ((1, 1), 10.0, True)


#### Exercise: 

Simulate one episode from A using a fixed policy (e.g., always DOWN if possible else RIGHT). Print trajectory and return.

V(s): How good is this state if I follow policy π from here?

Q(s,a): How good is taking action a in state s, then following π?